In [ ]:
from darts import TimeSeries
from pandas import Timedelta

from aare.AareDataset import AareDataset
from aare.params import read_params
from aare.preparation import (
    resample,
    remove_faulty_periods,
    remove_outliers,
    interpolate,
)
from aare.remote_existenz_store import RemoteExistenzStore
from aare.utils import to_ts

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
params = read_params()
store = RemoteExistenzStore()
ds = AareDataset(
    store, params["training"]["val_split"], params["training"]["test_split"]
)

In [ ]:
df = ds.get_val()
o_df = df.copy()
df

In [ ]:
df = resample(df)
df = remove_faulty_periods(df)
df = remove_outliers(df)
df = interpolate(df, drop_filled=True)

In [ ]:
df

In [ ]:
# must avoid NaNs, for real training use
# better imputation methods (or partial training)
df = df.ffill()

In [ ]:
ts = to_ts(df)
ts

In [ ]:
from darts.models import NaiveSeasonal

snaive = NaiveSeasonal(K=24)  # daily seasonality
snaive

In [ ]:
# test every day always with a horizon of 4 days using as information the 24 hours prior.
historical_forecasts = snaive.historical_forecasts(
    ts, stride=24, train_length=24, forecast_horizon=4 * 24, last_points_only=False
)
print(len(historical_forecasts))
type(historical_forecasts[0])

In [ ]:
from darts.metrics import mae, rmse

# MAE and RMSE are prob the most appropriate metrics (see eda and -> specs).
backtest = snaive.backtest(
    ts, historical_forecasts=historical_forecasts, metric=[mae, rmse]
)
backtest

In [ ]:
last_forecast: TimeSeries = historical_forecasts[-1]
ts[last_forecast.start_time() - Timedelta(24, "h") : last_forecast.end_time()].plot(
    label="actual"
)
last_forecast.plot(label="prediction")

In [ ]:
# If we don't use darts for forecasting: historical_forecasts is just a list of TS with forecast values (length of the horizon)
# so we can construct this ourselves and then still use the backtest function of darts to easily calculate the metrics.
# I think that should work. But hopefully there's a good global darts model that already works with the non-retrain mode of backtest.
# for global models, the train_length would be replaced by input_chunk_length on the model itself, I assume.
# Then you could use pre-trained mode (aka non-retrain).